# Laya UltraFast Decision Server for Jev-Ultrafast
Run the open-source **Laya (ModernBERT-large)** non-autoregressive decision model on **Google Colab's Free GPU/16GB RAM**.

This notebook serves sub-30ms browser decision predictions for `browser-use/jev-ultrafast` over a public HTTP tunnel, completely free without any proprietary TypeSafe API key or local RAM overhead.

In [ ]:
# 1. Install dependencies
!pip install -q laya pyngrok

In [ ]:
# 2. Authenticate ngrok (Optional: Get a free auth token at https://dashboard.ngrok.com/get-started/your-authtoken)
import os
from pyngrok import ngrok

NGROK_AUTHTOKEN = "" # Paste your token here if you have one
if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Open tunnel on port 8000
public_url = ngrok.connect(8000).public_url
print("=" * 65)
print("YOUR REMOTE LAYA ENDPOINT:", public_url)
print("=" * 65)
print(f"\nSet this in your local machine's terminal or .env:\n")
print(f"export LAYA_ENDPOINT={public_url}")

In [ ]:
# 3. Start the Laya Decision Engine HTTP Server
import json
import time
from http.server import HTTPServer, BaseHTTPRequestHandler
from laya import Router

print("Loading Laya Router on Colab GPU/CPU...")
router = Router()
print("Laya Router initialized and ready!")

class LayaHandler(BaseHTTPRequestHandler):
    def _send_json(self, status, data):
        body = json.dumps(data).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Access-Control-Allow-Origin", "*")
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        self._send_json(200, {"status": "ok", "service": "colab-laya", "engine": "laya-multilingual"})

    def do_POST(self):
        content_length = int(self.headers.get("Content-Length", 0))
        body = json.loads(self.rfile.read(content_length).decode("utf-8"))
        context = body.get("context", "")
        questions = body.get("questions", {})
        t0 = time.perf_counter()
        result = router.predict(context, questions)
        result["latency_ms"] = round((time.perf_counter() - t0) * 1000, 2)
        self._send_json(200, result)

server = HTTPServer(("0.0.0.0", 8000), LayaHandler)
print(f"Server running at {public_url} ... Serving requests!")
server.serve_forever()